# Visualizing Spatially Variable Genes with Squidpy

Reproducible code for the blog post at [spatiabio.com](https://www.spatiabio.com).

Dataset: 10x Genomics Visium mouse brain demo. This notebook visualizes the top 5 SVGs identified in the [Moran's I post](https://www.spatiabio.com/2026/06/spatially-variable-genes-with-squidpy.html).

## Setup

In [ ]:
import squidpy as sq
import scanpy as sc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

## Load and preprocess

Same pipeline as the SVG analysis post.

In [ ]:
adata = sq.datasets.visium_hne_adata()
# Or load from file: adata = sc.read_h5ad("visium_hne_adata.h5ad")

sc.pp.highly_variable_genes(adata, n_top_genes=500, flavor="cell_ranger")
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sq.gr.spatial_neighbors(adata)
print(f"Shape: {adata.shape}")

## Visualize top 5 SVGs

Top genes from Moran's I analysis (post 5):

| Rank | Gene | Moran's I |
|------|------|-----------|
| 1 | Itpka | 0.674 |
| 2 | Fezf1 | 0.634 |
| 3 | Baiap3 | 0.622 |
| 4 | Shox2 | 0.611 |
| 5 | Slc30a3 | 0.600 |

In [ ]:
top5 = ["Itpka", "Fezf1", "Baiap3", "Shox2", "Slc30a3"]

sq.pl.spatial_scatter(
    adata,
    color=top5,
    ncols=3,
    title=[f"{g}" for g in top5]
)

## Individual gene plots

In [ ]:
for gene in top5:
    sq.pl.spatial_scatter(
        adata,
        color=gene,
        title=f"{gene} (Moran's I spatial expression)"
    )

## Fix colormap scale for cross-gene comparison

> By default each gene scales independently. Use  to fix the range for fair comparison.

In [ ]:
sq.pl.spatial_scatter(adata, color="Itpka", vmin=0, vmax=20, title="Itpka (fixed scale)")

## Pattern summary

**Laminar patterns** (horizontal bands along cortical surface):
-  — deep cortical layer L5/L6
-  — broader cortical band

**Regional patterns** (compact clusters = discrete anatomical structures):
-  — hippocampus
-  — subcortical nucleus
-  — thalamus/hypothalamus

Both modes score high on Moran's I but for different reasons — laminar genes have many neighbors in the same band, regional genes have high on/off contrast against background.